<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 생성 모델 · 04. WGAN

## Wasserstein GAN

- **원 논문:** [Wasserstein GAN](https://proceedings.mlr.press/v70/arjovsky17a.html)
- **저자 / 발표:** Martin Arjovsky, Soumith Chintala, and Léon Bottou · ICML (2017)
- **난이도 / 예상 시간:** 중급 · 약 55분
- **선수 지식:** GAN, Lipschitz 함수, optimizer
- **로컬 학습 데이터:** `data/field_curriculum/generative_samples.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** 2차원 분포에서 sigmoid 없는 critic, n_critic 업데이트, weight clipping을 구현한다.

**축소하거나 생략한 부분:** 논문은 이미지 모델과 8-Gaussian toy를 사용한다. 여기서는 toy 분포로 Algorithm 1을 그대로 축소한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §2, Eq. (1) | Earth Mover distance | 겹치지 않는 분포에서도 연속적인 거리 신호가 생기는 이유를 연결한다. |
| §3, Eq. (2)–(3) | Kantorovich–Rubinstein critic | sigmoid 없는 scalar critic의 real-fake 차이를 최대화한다. |
| §3, Algorithm 1 | clipped critic algorithm | n_critic, RMSProp, parameter clipping 순서를 구현한다. |

## 핵심 재현

        Eq. (3)의 critic은 확률이 아니라 실수 점수를 냅니다. Algorithm 1처럼 critic을 여러 번
        갱신하고 가중치를 compact set으로 clip해 1-Lipschitz 제약을 근사합니다.

In [ ]:
from pathlib import Path
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(341)
np.random.seed(341)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
# These teaching batches are intentionally tiny; GPU launch/transfer
# overhead can dominate, so use AI_LAB_DEVICE=cpu for minimum wall time.

DATA_PATH = Path("data/field_curriculum/generative_samples.npz")
assert DATA_PATH.exists(), f"준비된 로컬 데이터가 없습니다: {DATA_PATH}"
raw = np.load(DATA_PATH)
expected = {"points", "point_labels", "images", "image_labels", "domain_a", "domain_b"}
assert expected.issubset(raw.files), f"NPZ 키 불일치: {sorted(raw.files)}"
points = torch.tensor(raw["points"], dtype=torch.float32)
point_labels = torch.tensor(raw["point_labels"], dtype=torch.long)
images = torch.tensor(raw["images"], dtype=torch.float32)
image_labels = torch.tensor(raw["image_labels"], dtype=torch.long)
domain_a = torch.tensor(raw["domain_a"], dtype=torch.float32)
domain_b = torch.tensor(raw["domain_b"], dtype=torch.float32)
points, point_labels, images, image_labels, domain_a, domain_b = ACCELERATOR.move(
    points, point_labels, images, image_labels, domain_a, domain_b
)


# Seeded random draws stay on CPU so CUDA, MPS, and CPU follow the same
# sampling stream; complete tensors are then transferred to the lab device.
def randn_device(size, *, generator):
    return torch.randn(size, generator=generator).to(DEVICE)


def rand_device(size, *, generator):
    return torch.rand(size, generator=generator).to(DEVICE)


def randint_device(high, size, *, generator):
    return torch.randint(high, size, generator=generator).to(DEVICE)


if float(images.min()) < 0.0:
    images_01 = ((images + 1.0) / 2.0).clamp(0.0, 1.0)
else:
    images_01 = images.clamp(0.0, 1.0)
assert points.shape == (600, 2) and images.shape == (300, 1, 8, 8)
assert domain_a.shape == domain_b.shape == (600, 2)
print(ACCELERATOR.summary())
print("local data:", tuple(points.shape), tuple(images.shape), tuple(domain_a.shape))

## 포트폴리오 구현 설계: 수식 → 책임 → 검증

논문의 핵심 목적을 실행 가능한 객체의 `forward`, `loss`, `update`, `evaluate`로
나눕니다. 아래 식이 클래스 내부에서 생략되지 않고 어떤 tensor 연산이 되는지
메서드별로 추적해 보세요.

$$
W(P_r,P_g)\approx\max_{\lVert f\rVert_L\le1}\mathbb E_{x\sim P_r}[f(x)]-\mathbb E_{z}[f(G(z))]
$$

- **기호와 역할:** $f$는 sigmoid가 없는 critic, $P_r/P_g$는 real/generated 분포, $\lVert f\rVert_L\le1$은 Lipschitz 제약입니다.
- **shape 계약:** `real/fake [N, 2] → critic score [N, 1]`
- **논문 위치:** `§2, Eq. (1)`의 **Earth Mover distance**
- **코드 Task:** critic gap, n_critic 반복, RMSProp, parameter clipping 순서를 구현합니다.
- **학습·평가 흐름:** critic 3회 update+clip → generator 1회 update → critic gap 평가
- **원문 대비 한계:** 논문은 이미지 모델과 8-Gaussian toy를 사용한다. 여기서는 toy 분포로 Algorithm 1을 그대로 축소한다.
- **구현 이유:** 수식의 tensor 경계와 optimizer 책임을 클래스 메서드별로
  분리해 코드와 논문을 한 줄씩 대조할 수 있게 합니다.
- **완료 증거:** 마지막 셀에서 `update`, parameter 변화, 평가 metric을
  모두 `assert`합니다.

정답 클래스는 교육용 편의 함수가 핵심 계산을 대신하지 않도록 loss와 update 순서를
메서드 본문에 드러냅니다. 실습본에서는 같은 공개 API를 유지한 채 TODO를 채우세요.

In [ ]:
class WassersteinGanLab(nn.Module):
    """WGAN의 핵심 학습·평가 경로. 입출력 계약: real/fake [N, 2] → critic score [N, 1]."""

    def __init__(self, generator, critic, generator_optimizer, critic_optimizer):
        """구성 요소와 optimizer를 저장한다. 학습은 update에서만 수행한다."""
        super().__init__()
        self.generator = generator
        self.critic = critic
        self.generator_optimizer = generator_optimizer
        self.critic_optimizer = critic_optimizer
        self.clip_value = 0.05

    def config(self):
        """재현 범위와 핵심 하이퍼파라미터를 dict로 반환한다."""
        return {"noise_dim": 2, "critic_steps": 3, "clip": self.clip_value}

    def forward(self, noise):
        """논문 순전파를 계산한다. shape 계약: real/fake [N, 2] → critic score [N, 1]."""
        fake = self.generator(noise)
        return (fake, self.critic(fake))

    def loss(self, real, noise):
        """논문의 목적 함수를 미분 가능한 scalar tensor로 반환한다."""
        fake, fake_score = self.forward(noise)
        real_score = self.critic(real)
        critic_objective = fake_score.mean() - real_score.mean()
        generator_objective = -self.critic(fake).mean()
        return (critic_objective, generator_objective)

    def update(self, real, critic_noises, generator_noise):
        """zero_grad, backward, step을 수행하고 유한한 float loss를 반환한다."""
        critic_value = None
        for noise in critic_noises:
            fake = self.generator(noise).detach()
            critic_value = self.critic(fake).mean() - self.critic(real).mean()
            self.critic_optimizer.zero_grad()
            critic_value.backward()
            self.critic_optimizer.step()
            with torch.no_grad():
                for parameter in self.critic.parameters():
                    parameter.clamp_(-self.clip_value, self.clip_value)
        generated = self.generator(generator_noise)
        generator_value = -self.critic(generated).mean()
        self.generator_optimizer.zero_grad()
        generator_value.backward()
        self.generator_optimizer.step()
        return (float(critic_value.detach()), float(generator_value.detach()))

    def evaluate(self, real, noise):
        """no_grad 경로에서 논문과 연결된 정량 지표 dict를 반환한다."""
        with torch.no_grad():
            fake, fake_score = self.forward(noise)
            gap = self.critic(real).mean() - fake_score.mean()
        return {"critic_gap": float(gap), "fake_mean": fake.mean(dim=0).tolist()}

### 단계 2. 핵심 연산 검증

- **논문의 어느 부분인가:** `§3, Eq. (2)–(3)` — **Kantorovich–Rubinstein critic**
- **구현 이유:** sigmoid 없는 scalar critic의 real-fake 차이를 최대화한다.
- **읽을 코드:** 아래 `implementation` 셀에서 중간 tensor의 shape, gradient가 흐르는
  경로와 `detach` 또는 `no_grad` 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
G = nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 2)).to(DEVICE)
C = nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 1)).to(DEVICE)


def critic_loss(real_score, fake_score):
    return fake_score.mean() - real_score.mean()


def clip_critic(model, c=0.05):
    with torch.no_grad():
        for p in model.parameters():
            p.clamp_(-c, c)


probe = critic_loss(
    torch.tensor([2.0, 3.0], device=DEVICE),
    torch.tensor([-1.0, 0.0], device=DEVICE),
)
assert torch.allclose(probe, torch.tensor(-3.0, device=DEVICE))

### 단계 3. 학습·업데이트

- **논문의 어느 부분인가:** `§3, Algorithm 1` — **clipped critic algorithm**
- **구현 이유:** n_critic, RMSProp, parameter clipping 순서를 구현한다.
- **읽을 코드:** 아래 `training` 셀에서 중간 tensor의 shape, gradient가 흐르는
  경로와 `detach` 또는 `no_grad` 경계를 확인합니다.

실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
주석으로 적은 뒤 실습하세요.
- **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

In [ ]:
oc = torch.optim.RMSprop(C.parameters(), lr=0.003)
og = torch.optim.RMSprop(G.parameters(), lr=0.003)
rng = torch.Generator().manual_seed(342)
history = []
fixed_z = randn_device((256, 2), generator=torch.Generator().manual_seed(343))

portfolio_model = WassersteinGanLab(G, C, og, oc)
initial_parameters = torch.cat(
    [parameter.detach().flatten().cpu() for parameter in G.parameters()]
)
for _ in range(140):
    real = points[randint_device(len(points), (96,), generator=rng)]
    critic_noises = [randn_device((96, 2), generator=rng) for _ in range(3)]
    generator_noise = randn_device((96, 2), generator=rng)
    history.append(portfolio_model.update(real, critic_noises, generator_noise))
final_parameters = torch.cat(
    [parameter.detach().flatten().cpu() for parameter in G.parameters()]
)
parameter_delta = float((final_parameters - initial_parameters).abs().sum())
assert parameter_delta > 0.0 and np.isfinite(history).all()

### 단계 4. 평가·시각화

        - **논문의 어느 부분인가:** `§2, Eq. (1)` — **Earth Mover distance**
        - **구현 이유:** 겹치지 않는 분포에서도 연속적인 거리 신호가 생기는 이유를 연결한다.
        - **읽을 코드:** 아래 `evaluation` 셀에서 중간 tensor의 shape, gradient가 흐르는
          경로와 `detach` 또는 `no_grad` 경계를 확인합니다.

        실행 결과만 보지 말고, 이 단계의 입력과 출력이 앞선 수식의 어떤 기호인지 먼저
        주석으로 적은 뒤 실습하세요.
        - **완료 증거:** 아래 셀의 `assert`와 출력 metric이 shape·loss 계약을 검사합니다.

- **평가 범위:** 이 값은 작은 학습 batch의 메커니즘 진단이며, held-out 생성 품질이나 원 논문의 benchmark 성능으로 해석하지 않습니다.

In [ ]:
portfolio_metrics = portfolio_model.evaluate(points, fixed_z)
assert portfolio_metrics

with torch.no_grad():
    fake = G(fixed_z)
    gap = float(C(points).mean() - C(fake).mean())
print(f"estimated Wasserstein critic gap={gap:.4f}")
assert math.isfinite(gap)
h = np.asarray(history)
points_cpu, fake_cpu = points.detach().cpu(), fake.detach().cpu()
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].scatter(points_cpu[:, 0], points_cpu[:, 1], s=6, alpha=0.3, label="real")
axes[0].scatter(fake_cpu[:, 0], fake_cpu[:, 1], s=7, alpha=0.4, label="fake")
axes[0].legend()
axes[1].plot(h[:, 0], label="critic gap")
axes[1].plot(h[:, 1], label="G objective")
axes[1].legend()
fig.tight_layout()
plt.show()

KR dual의 critic은 확률 분류기가 아니라 Lipschitz 실함수입니다. 값의 차이가 Wasserstein 거리 추정량이므로 [0,1] 제약이 없습니다.

### 단계 5. 통합 클래스가 실제 update와 evaluate를 소유하는지 검증

- **논문 위치:** `§2, Eq. (1)`의 **Earth Mover distance**
- **핵심 식:**

  $$
  W(P_r,P_g)\approx\max_{\lVert f\rVert_L\le1}\mathbb E_{x\sim P_r}[f(x)]-\mathbb E_{z}[f(G(z))]
  $$

- **입출력 shape:** `real/fake [N, 2] → critic score [N, 1]`
- **구현 이유:** 앞 셀은 수식을 작은 연산으로 분해해 확인하고,
  이 셀은 같은 구성 요소를 `WassersteinGanLab`에 주입해
  최종 학습·평가 경로를 하나로 묶습니다. 겹치지 않는 분포에서도 연속적인 거리 신호가 생기는 이유를 연결한다.
- **완료 증거:** `update(...)`가 유한한 loss를 반환하고,
  `parameter_delta > 0`이며, `evaluate(...)`의 논문 대응 지표가 유한해야 합니다.
  `config()`만 호출하는 것은 실행 재현으로 보지 않습니다.

In [ ]:
config_contract = portfolio_model.config()
numeric_metrics = [
    float(value)
    for value in portfolio_metrics.values()
    if isinstance(value, (int, float))
]
assert config_contract and len(history) > 0
assert np.isfinite(np.asarray(history, dtype=float)).all()
assert parameter_delta > 0.0
assert numeric_metrics and np.isfinite(numeric_metrics).all()
print(
    {
        "class": type(portfolio_model).__name__,
        "steps": len(history),
        "parameter_delta": parameter_delta,
        "metrics": portfolio_metrics,
    }
)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 논문은 이미지 모델과 8-Gaussian toy를 사용한다. 여기서는 toy 분포로 Algorithm 1을 그대로 축소한다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.